In [1]:
import Student
!pip install wilds matplotlib torch torchvision terratorch umap-learn seaborn scikit-learn matplotlib terratorch

In [2]:

import ClusteringPipeline
# ---------------------------------------------------------
#  Simplified DataLoader
# ---------------------------------------------------------
import torch
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
from terratorch.models.backbones.terramind.model.terramind_register import v1_pretraining_mean, v1_pretraining_std

my_root_path = "./data/"
def get_input(idx):
        """
        Returns x for a given idx.
        """
        img = Image.open(Path(my_root_path + 'fmow_v1.1') / 'images' / f'rgb_img_{idx}.png').convert('RGB')
        return img



# 1. Fetch the 8-bit RGB specific statistics
tm_rgb_mean = v1_pretraining_mean['untok_sen2rgb@224']
tm_rgb_std = v1_pretraining_std['untok_sen2rgb@224']
"""
# 2. Scale the stats down to 0.0-1.0 to match T.ToTensor()
#SCALED_MEAN = [x / 255.0 for x in tm_rgb_mean]
#SCALED_STD = [x / 255.0 for x in tm_rgb_std]
print(f"Scaled RGB Mean: {SCALED_MEAN}")
print(f"Scaled RGB Std:  {SCALED_STD}")

# 3. Create the Transform Pipeline
terramind_transform = T.Compose([
    T.ToTensor(),  # Scales 0-255 pixels down to 0.0-1.0
    T.Normalize(mean=SCALED_MEAN, std=SCALED_STD) # Perfectly standardizes the data
])
"""
terramind_transform = T.Compose([
    T.PILToTensor(),                           # Keeps pixels 0-255, outputs torch.uint8
    T.Lambda(lambda x: x.to(torch.float32)),   # Converts to float32 WITHOUT scaling to 0.0-1.0
    T.Normalize(mean=tm_rgb_mean, std=tm_rgb_std) # Normalizes using the raw 0-255 stats
])





C:\Users\kevin\PycharmProjects\master_thesis\.venv\Lib\site-packages\outdated\__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
C:\Users\kevin\PycharmProjects\master_thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch.nn as nn
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, num_classes=6):
        super(SimpleMLP, self).__init__()
        self.layer_stack = nn.Sequential(
            # Input to Hidden
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            # Optional: Dropout helps significantly with 64 samples/class
            nn.Dropout(0.3),
            # Hidden to Output
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.layer_stack(x)
class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes=62):
        super(LinearProbe, self).__init__()
        # Nessun hidden layer, solo una proiezione diretta
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, num_classes),
        )

    def forward(self, x):
        return self.classifier(x)

In [4]:
import FoundationModel
# 1. Setup the pipeline and get full raw metadata
pipeline = ClusteringPipeline.ClusteringPipeline()
df_full = pipeline.get_dataset()
fm_model_large = FoundationModel.FoundationModel(name='terramind_v1_large')
df_sample = pipeline.sample_dataset(df_full, samples_per_class=50)
df_train_large = pipeline.get_embeddings(df_sample, fm_model_large, get_input, terramind_transform,save_path="./large")

embedding_dim_large = len(df_train_large['embedding'].iloc[0])
'''
fm_model_small = FoundationModel.FoundationModel(name='terramind_v1_small')
fm_model_tiny = FoundationModel.FoundationModel(name='terramind_v1_tiny')

df_sample = pipeline.sample_dataset(df_full, samples_per_class=64)

df_train_base = pipeline.get_embeddings(df_sample, fm_model_base, get_input, terramind_transform,save_path="./base")
df_train_base = pipeline.cluster_by_class(df_train_base, num_clusters=6)

df_train_small = pipeline.get_embeddings(df_sample, fm_model_small, get_input, terramind_transform,save_path="./small")
df_train_small = pipeline.cluster_by_class(df_train_small, num_clusters=6)

df_train_tiny = pipeline.get_embeddings(df_sample, fm_model_tiny, get_input, terramind_transform,save_path = "./tiny")
df_train_tiny = pipeline.cluster_by_class(df_train_tiny, num_clusters=6)

embedding_dim_base = len(df_train_base['embedding'].iloc[0])
embedding_dim_small = len(df_train_small['embedding'].iloc[0])
embedding_dim_tiny = len(df_train_tiny['embedding'].iloc[0])
'''


2026-05-06 17:21:58,865 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-esa-geospatial/TerraMind-1.0-large/resolve/main/TerraMind_v1_large.pt "HTTP/1.1 302 Found"


--- Sampling 50 images per class ---
Extracting features...


Extracting: 100%|██████████| 25/25 [10:38<00:00, 25.54s/it]

Saving new embeddings dataset to disk...

Calculated Dataset Mean: [0.41779667139053345, 0.4229981303215027, 0.3962430953979492]
Calculated Dataset Std:  [0.2522008419036865, 0.2463635355234146, 0.2511313259601593]


'\nfm_model_small = FoundationModel.FoundationModel(name=\'terramind_v1_small\')\nfm_model_tiny = FoundationModel.FoundationModel(name=\'terramind_v1_tiny\')\n\ndf_sample = pipeline.sample_dataset(df_full, samples_per_class=64)\n\ndf_train_base = pipeline.get_embeddings(df_sample, fm_model_base, get_input, terramind_transform,save_path="./base")\ndf_train_base = pipeline.cluster_by_class(df_train_base, num_clusters=6)\n\ndf_train_small = pipeline.get_embeddings(df_sample, fm_model_small, get_input, terramind_transform,save_path="./small")\ndf_train_small = pipeline.cluster_by_class(df_train_small, num_clusters=6)\n\ndf_train_tiny = pipeline.get_embeddings(df_sample, fm_model_tiny, get_input, terramind_transform,save_path = "./tiny")\ndf_train_tiny = pipeline.cluster_by_class(df_train_tiny, num_clusters=6)\n\nembedding_dim_base = len(df_train_base[\'embedding\'].iloc[0])\nembedding_dim_small = len(df_train_small[\'embedding\'].iloc[0])\nembedding_dim_tiny = len(df_train_tiny[\'embedding

In [5]:
mlp_model = LinearProbe(input_dim=embedding_dim_large, num_classes=62)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-3)

# Train using the newly created macro_class column
history = pipeline.train_offline(
    model=mlp_model,
    df=df_train_large,
    criterion=criterion,
    optimizer=optimizer,
    target_col='category',
    epochs=100
)

Preparing data for training on target: category
Epoch [1/100] | Train Acc: 8.2% | Val Acc: 12.6%
Epoch [5/100] | Train Acc: 32.6% | Val Acc: 21.5%
Epoch [10/100] | Train Acc: 45.6% | Val Acc: 23.4%
Epoch [15/100] | Train Acc: 54.1% | Val Acc: 23.7%
Epoch [20/100] | Train Acc: 61.2% | Val Acc: 25.3%
Epoch [25/100] | Train Acc: 66.6% | Val Acc: 24.4%
Epoch [30/100] | Train Acc: 72.0% | Val Acc: 24.4%
Epoch [35/100] | Train Acc: 75.4% | Val Acc: 23.1%
Epoch [40/100] | Train Acc: 78.5% | Val Acc: 24.8%
Epoch [45/100] | Train Acc: 81.8% | Val Acc: 24.5%
Epoch [50/100] | Train Acc: 84.3% | Val Acc: 23.5%
Epoch [55/100] | Train Acc: 86.4% | Val Acc: 23.1%
Epoch [60/100] | Train Acc: 88.5% | Val Acc: 23.1%
Epoch [65/100] | Train Acc: 89.6% | Val Acc: 23.4%
Epoch [70/100] | Train Acc: 91.5% | Val Acc: 23.5%
Epoch [75/100] | Train Acc: 93.3% | Val Acc: 23.1%
Epoch [80/100] | Train Acc: 94.2% | Val Acc: 23.7%
Epoch [85/100] | Train Acc: 95.5% | Val Acc: 23.4%
Epoch [90/100] | Train Acc: 96.4% | V

In [6]:
mlp_model = LinearProbe(input_dim=embedding_dim_small, num_classes=6)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-3)

# Train using the newly created macro_class column
history = pipeline.train_offline(
    model=mlp_model,
    df=df_train_small,
    criterion=criterion,
    optimizer=optimizer,
    target_col='macro_class',
    epochs=100
)

NameError: name 'embedding_dim_small' is not defined

In [ ]:
mlp_model = LinearProbe(input_dim=embedding_dim_tiny, num_classes=62)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-3)

# Train using the newly created macro_class column
history = pipeline.train_offline(
    model=mlp_model,
    df=df_train_tiny,
    criterion=criterion,
    optimizer=optimizer,
    target_col='category',
    epochs=100
)

In [ ]:
mlp_model = LinearProbe(input_dim=embedding_dim_tiny, num_classes=6)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-3)

# Train using the newly created macro_class column
history = pipeline.train_offline(
    model=mlp_model,
    df=df_train_tiny,
    criterion=criterion,
    optimizer=optimizer,
    target_col='macro_class',
    epochs=100
)

In [ ]:
mlp_model = SimpleMLP(input_dim=embedding_dim_small, num_classes=6)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-3)

# Train using the newly created macro_class column
history = pipeline.train_offline(
    model=mlp_model,
    df=df_train_small,
    criterion=criterion,
    optimizer=optimizer,
    target_col='macro_class',
    epochs=100
)

In [ ]:
mlp_model = SimpleMLP(input_dim=embedding_dim_tiny, num_classes=62)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-2)

# Train using the newly created macro_class column
history = pipeline.train_offline(
    model=mlp_model,
    df=df_train_tiny,
    criterion=criterion,
    optimizer=optimizer,
    target_col='category',
    epochs=100
)

In [ ]:
mlp_model = LinearProbe(input_dim=embedding_dim_base, num_classes=6)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-3)

# Train using the newly created macro_class column
history = pipeline.train_offline(
    model=mlp_model,
    df=df_train_base,
    criterion=criterion,
    optimizer=optimizer,
    target_col='macro_class',
    epochs=100
)

In [ ]:
mlp_model = LinearProbe(input_dim=embedding_dim_base, num_classes=62)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-3)

# Train using the newly created macro_class column
history = pipeline.train_offline(
    model=mlp_model,
    df=df_train_base,
    criterion=criterion,
    optimizer=optimizer,
    target_col='category',
    epochs=100
)

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
resnet_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD)
])



In [ ]:
import gc
import os

import torch


from tqdm import tqdm
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from wilds import get_dataset

# Use sklearn's LabelEncoder for tabular dataframe columns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def train_offline_resnet(model, df, criterion, optimizer, target_col='category', epochs=100, batch_size=32, log_interval=1):
    """Builds DataLoaders directly from the unified dataframe and trains the model."""
    print(f"Preparing data for training on target: {target_col}")
    # 1. Prepare Data
    X = df['index'].to_numpy()
    le = LabelEncoder()
    y = le.fit_transform(df[target_col])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.long),
                                                torch.tensor(y_train, dtype=torch.long)),
                                  batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.long),
                                               torch.tensor(y_test, dtype=torch.long)),
                                 batch_size=batch_size, shuffle=False)

    # 2. Train Loop
    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0.0, 0, 0

        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X, batch_y
            imgs = []
            for idx in batch_X:
                img = get_input(idx)
                imgs.append(resnet_transform(img))
            input = torch.stack(imgs,dim=0)
            optimizer.zero_grad()
            outputs = model(input)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            predicted = torch.argmax(outputs, dim=1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            del input,outputs,imgs
            gc.collect()
        train_acc = 100 * correct / total
        train_loss /= len(train_loader)

        val_loss, val_acc = evaluate_resnet(model, test_loader, criterion)

        history['train_loss'].append(train_loss)
        history['test_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(val_acc)

        if (epoch + 1) % log_interval == 0 or epoch == 0:
            print(f"Epoch [{epoch + 1}/{epochs}] | Train Acc: {train_acc:.1f}% | Val Acc: {val_acc:.1f}%")

    print("\n--- Final Evaluation ---")
    final_test_loss, final_test_acc = evaluate_resnet(model, test_loader, criterion)
    print(f"Final Test Accuracy: {final_test_acc:.2f}%")
    return history

def evaluate_resnet(model, loader, criterion):
    """Internal helper for evaluation."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X, batch_y
            imgs = []
            for idx in batch_X:
                img = get_input(idx)
                imgs.append(resnet_transform(img))
            input = torch.stack(imgs,dim=0)
            outputs = model(input)
            loss = criterion(outputs, batch_y)

            running_loss += loss.item()
            predicted = torch.argmax(outputs, dim=1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

    return running_loss / len(loader), 100 * correct / total


In [ ]:
import Student
model = Student.Student(numberOfClasses=62)
criterion = nn.CrossEntropyLoss()
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3)
train_offline_resnet(model = model,df= df_sample,criterion=criterion,optimizer=optimizer,target_col='category',epochs=100)

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print(f"Parametri totali del modello: {all_params:,}")
print(f"Parametri effettivamente in addestramento: {trainable_params:,}")

In [ ]:
pipeline.visualize_category_images(df = df_sample,category_name="airport_terminal",get_input_fn=get_input,num_images=5)